In [ ]:
# Main 2xT4 Stream1 MLP native-vs-PyTorch benchmark config
GITHUB_REPO_URL = "https://github.com/TryDotAtwo/MultiGPUBeamSearch.git"
GITHUB_REF = "stream1-mlp-torch-kaggle-v2-code"
GITHUB_EXPECTED_COMMIT = "cc90a3d"
CUDA_ARCHITECTURES = "75"
BENCH_GPUS = [0, 1]
BENCH_PUZZLE_ID = 0
NATIVE_WEIGHT_SUBDIR = "stream1_weights"
TORCH_ROWS = "2048,4096,8192,16384,32768,65536,131072"
BENCH_REPORT_DIR = "/kaggle/working/mlp_native_vs_torch_reports"
BENCH_LOG_DIR = "/kaggle/working/mlp_native_vs_torch_logs"
NATIVE_ROWS_CSV = "/kaggle/working/native_mlp_benchmark_rows.csv"
TORCH_ROWS_CSV = "/kaggle/working/torch_mlp_benchmark_rows.csv"
COMPARISON_CSV = "/kaggle/working/mlp_native_vs_torch_comparison.csv"


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

WORK_DIR = Path('/kaggle/working')
TMP_DIR = Path('/tmp')
REPO_DIR = TMP_DIR / 'beam_solver_mlp_benchmark'
CUTLASS_DIR = TMP_DIR / 'cutlass'
BUILD_DIR = TMP_DIR / 'beam_build_mlp_benchmark'
BENCH_REPORT_DIR = Path(BENCH_REPORT_DIR)
BENCH_LOG_DIR = Path(BENCH_LOG_DIR)
BENCH_REPORT_DIR.mkdir(parents=True, exist_ok=True)
BENCH_LOG_DIR.mkdir(parents=True, exist_ok=True)


def run_checked(cmd, cwd=None, env=None):
    cmd = [str(part) for part in cmd]
    print('+ ' + ' '.join(cmd), flush=True)
    subprocess.run(cmd, cwd=cwd, env=env, check=True)


def run_capture(cmd, cwd=None, env=None, check=True):
    cmd = [str(part) for part in cmd]
    print('+ ' + ' '.join(cmd), flush=True)
    result = subprocess.run(cmd, cwd=cwd, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout, end='', flush=True)
    if check and result.returncode != 0:
        raise subprocess.CalledProcessError(result.returncode, cmd, output=result.stdout)
    return result


def cleanup_path(path):
    path = Path(path)
    if path.exists():
        if path.is_dir():
            shutil.rmtree(path)
        else:
            path.unlink()


def disk_line(path):
    usage = shutil.disk_usage(path)
    return f'{path}: free={usage.free} total={usage.total}'

print('GITHUB_REF=', GITHUB_REF, flush=True)
print('GITHUB_EXPECTED_COMMIT=', GITHUB_EXPECTED_COMMIT, flush=True)
print('CUDA_ARCHITECTURES=', CUDA_ARCHITECTURES, flush=True)
print('disk_tmp=', disk_line('/tmp'), flush=True)
print('disk_working=', disk_line('/kaggle/working'), flush=True)
run_capture(['nvidia-smi'], check=False)
import torch
print('torch_version=', torch.__version__, flush=True)
print('torch_cuda_device_count=', torch.cuda.device_count(), flush=True)
if torch.cuda.device_count() < len(BENCH_GPUS):
    raise RuntimeError(f'expected at least {len(BENCH_GPUS)} CUDA devices, found {torch.cuda.device_count()}')

for path in (REPO_DIR, BUILD_DIR):
    cleanup_path(path)
if not CUTLASS_DIR.exists():
    run_checked(['git', 'clone', '--depth', '1', 'https://github.com/NVIDIA/cutlass.git', CUTLASS_DIR])
run_checked(['git', 'clone', '--branch', GITHUB_REF, '--depth', '1', GITHUB_REPO_URL, REPO_DIR])
commit = run_capture(['git', 'rev-parse', '--short', 'HEAD'], cwd=REPO_DIR).stdout.strip()
print('GITHUB_COMMIT=', commit, flush=True)
if GITHUB_EXPECTED_COMMIT and not commit.startswith(GITHUB_EXPECTED_COMMIT):
    raise RuntimeError(f'expected GitHub commit {GITHUB_EXPECTED_COMMIT}, got {commit}')

run_checked([
    'cmake', '-S', REPO_DIR, '-B', BUILD_DIR, '-G', 'Ninja',
    f'-DCUTLASS_DIR={CUTLASS_DIR}',
    f'-DBEAM_CUDA_ARCHITECTURES={CUDA_ARCHITECTURES}',
    '-DBEAM_ENABLE_DEBUG=OFF',
])
run_checked(['cmake', '--build', BUILD_DIR, '--target', 'stream_benchmark', '-j', '2'])
print('stream_benchmark_build_done=1', flush=True)


In [ ]:
import csv
import os
from pathlib import Path
import re
import subprocess
import time

native_pattern = re.compile(
    r'stream1_micro\s+'
    r'b_micro=(?P<b_micro>\d+)\s+'
    r'concurrent=(?P<concurrency>\d+)\s+'
    r'(?:rows_per_launch_group=(?P<rows>\d+)\s+)?'
    r'ms_per_launch_group=(?P<ms>[0-9.]+)\s+'
    r'parents_per_sec=(?P<parents>[0-9.]+)\s+'
    r'candidates_per_sec=(?P<candidates>[0-9.]+)'
    r'(?:\s+scratch_bytes=(?P<scratch>\d+))?'
)
native_rows = []
for gpu in BENCH_GPUS:
    report_path = BENCH_REPORT_DIR / f'native_mlp_benchmark_gpu{gpu}.md'
    log_path = BENCH_LOG_DIR / f'native_mlp_benchmark_gpu{gpu}.log'
    env = os.environ.copy()
    env.update({
        'CUDA_VISIBLE_DEVICES': str(gpu),
        'BEAM_WEIGHT_DIR': str(REPO_DIR / NATIVE_WEIGHT_SUBDIR),
        'BEAM_STREAM1_ONLY': '1',
        'BEAM_STREAM_BENCH_REPORT': str(report_path),
    })
    cmd = [str(BUILD_DIR / 'stream_benchmark'), str(BENCH_PUZZLE_ID)]
    print('RUN_NATIVE_MLP_BENCH_START', {'gpu': gpu, 'cmd': cmd, 'report': str(report_path), 'log': str(log_path)}, flush=True)
    start = time.time()
    with log_path.open('w', buffering=1, encoding='utf-8') as log:
        proc = subprocess.Popen(cmd, cwd=REPO_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        assert proc.stdout is not None
        for line in proc.stdout:
            log.write(line)
            print(line, end='', flush=True)
            match = native_pattern.search(line)
            if match:
                gd = match.groupdict()
                b_micro = int(gd['b_micro'])
                concurrency = int(gd['concurrency'])
                rows = int(gd['rows']) if gd.get('rows') else b_micro * concurrency
                row = {
                    'gpu': gpu,
                    'implementation': 'native_cutlass_mlp',
                    'b_micro': b_micro,
                    'concurrency': concurrency,
                    'rows_per_launch_group': rows,
                    'ms_per_launch_group': float(gd['ms']),
                    'parents_per_sec': float(gd['parents']),
                    'candidates_per_sec': float(gd['candidates']),
                    'scratch_bytes': int(gd['scratch']) if gd.get('scratch') else '',
                    'status': 'pass',
                }
                native_rows.append(row)
        rc = proc.wait()
    elapsed = time.time() - start
    print('RUN_NATIVE_MLP_BENCH_DONE', {'gpu': gpu, 'rc': rc, 'seconds': elapsed, 'rows': len([r for r in native_rows if r['gpu'] == gpu])}, flush=True)
    if rc != 0:
        raise subprocess.CalledProcessError(rc, cmd)

native_csv = Path(NATIVE_ROWS_CSV)
with native_csv.open('w', newline='', encoding='utf-8') as fh:
    writer = csv.DictWriter(fh, fieldnames=['gpu', 'implementation', 'b_micro', 'concurrency', 'rows_per_launch_group', 'ms_per_launch_group', 'parents_per_sec', 'candidates_per_sec', 'scratch_bytes', 'status'])
    writer.writeheader()
    writer.writerows(native_rows)
print('NATIVE_MLP_BENCH_CSV=', native_csv, flush=True)
for gpu in BENCH_GPUS:
    best = max([r for r in native_rows if r['gpu'] == gpu], key=lambda r: r['candidates_per_sec'])
    print('NATIVE_MLP_BEST_GPU', gpu, best, flush=True)


In [ ]:
import csv
import os
from pathlib import Path
import subprocess
import sys
import time

torch_rows = []
for gpu in BENCH_GPUS:
    out_csv = BENCH_REPORT_DIR / f'torch_mlp_benchmark_gpu{gpu}.csv'
    log_path = BENCH_LOG_DIR / f'torch_mlp_benchmark_gpu{gpu}.log'
    env = os.environ.copy()
    env.update({'CUDA_VISIBLE_DEVICES': str(gpu)})
    cmd = [
        sys.executable,
        str(REPO_DIR / 'tools' / 'stream1_mlp_torch_benchmark.py'),
        '--weight-dir', str(REPO_DIR / NATIVE_WEIGHT_SUBDIR),
        '--test-csv', str(REPO_DIR / 'data' / 'test.csv'),
        '--puzzle-id', str(BENCH_PUZZLE_ID),
        '--gpu-label', str(gpu),
        '--rows', TORCH_ROWS,
        '--out-csv', str(out_csv),
    ]
    print('RUN_TORCH_MLP_BENCH_START', {'gpu': gpu, 'cmd': cmd, 'log': str(log_path), 'csv': str(out_csv)}, flush=True)
    start = time.time()
    with log_path.open('w', buffering=1, encoding='utf-8') as log:
        proc = subprocess.Popen(cmd, cwd=REPO_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        assert proc.stdout is not None
        for line in proc.stdout:
            log.write(line)
            print(line, end='', flush=True)
        rc = proc.wait()
    elapsed = time.time() - start
    print('RUN_TORCH_MLP_BENCH_DONE', {'gpu': gpu, 'rc': rc, 'seconds': elapsed}, flush=True)
    if rc != 0:
        raise subprocess.CalledProcessError(rc, cmd)
    with out_csv.open('r', encoding='utf-8', newline='') as fh:
        torch_rows.extend(list(csv.DictReader(fh)))

torch_csv = Path(TORCH_ROWS_CSV)
with torch_csv.open('w', newline='', encoding='utf-8') as fh:
    writer = csv.DictWriter(fh, fieldnames=['gpu', 'implementation', 'rows_per_launch_group', 'ms_per_launch_group', 'parents_per_sec', 'candidates_per_sec', 'status'])
    writer.writeheader()
    writer.writerows(torch_rows)
print('TORCH_MLP_BENCH_CSV=', torch_csv, flush=True)
for gpu in BENCH_GPUS:
    rows = [r for r in torch_rows if int(r['gpu']) == gpu and r['status'] == 'pass']
    best = max(rows, key=lambda r: float(r['candidates_per_sec']))
    print('TORCH_MLP_BEST_GPU', gpu, best, flush=True)


In [ ]:
import csv
from pathlib import Path

native_rows = list(csv.DictReader(open(NATIVE_ROWS_CSV, newline='', encoding='utf-8')))
torch_rows = list(csv.DictReader(open(TORCH_ROWS_CSV, newline='', encoding='utf-8')))
comparison_rows = []
for gpu in BENCH_GPUS:
    native_best = max(
        [r for r in native_rows if int(r['gpu']) == gpu and r['status'] == 'pass'],
        key=lambda r: float(r['candidates_per_sec']),
    )
    torch_best = max(
        [r for r in torch_rows if int(r['gpu']) == gpu and r['status'] == 'pass'],
        key=lambda r: float(r['candidates_per_sec']),
    )
    native_cps = float(native_best['candidates_per_sec'])
    torch_cps = float(torch_best['candidates_per_sec'])
    comparison = {
        'gpu': gpu,
        'native_b_micro': native_best['b_micro'],
        'native_concurrency': native_best['concurrency'],
        'native_rows_per_launch_group': native_best['rows_per_launch_group'],
        'native_candidates_per_sec': f'{native_cps:.1f}',
        'torch_rows_per_launch_group': torch_best['rows_per_launch_group'],
        'torch_candidates_per_sec': f'{torch_cps:.1f}',
        'native_over_torch': f'{native_cps / torch_cps:.4f}',
        'torch_over_native': f'{torch_cps / native_cps:.4f}',
    }
    comparison_rows.append(comparison)
    print('MLP_NATIVE_VS_TORCH_BEST', comparison, flush=True)
aggregate_native = sum(float(r['native_candidates_per_sec']) for r in comparison_rows)
aggregate_torch = sum(float(r['torch_candidates_per_sec']) for r in comparison_rows)
print('MLP_NATIVE_AGG_CANDIDATES_PER_SEC=', aggregate_native, flush=True)
print('MLP_TORCH_AGG_CANDIDATES_PER_SEC=', aggregate_torch, flush=True)
print('MLP_NATIVE_OVER_TORCH_AGG=', aggregate_native / aggregate_torch, flush=True)
comparison_csv = Path(COMPARISON_CSV)
with comparison_csv.open('w', newline='', encoding='utf-8') as fh:
    writer = csv.DictWriter(fh, fieldnames=list(comparison_rows[0].keys()))
    writer.writeheader()
    writer.writerows(comparison_rows)
print('MLP_COMPARISON_CSV=', comparison_csv, flush=True)
